# Connection failure and durable receipts

Dead end: passing the timeout as a PostgreSQL startup parameter failed at the connection pooler. A safe diagnostic exposed only the category startup_parameter. The corrected path applies SET LOCAL inside the transaction and preserves read-only release access.


In [ ]:
import json, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'evaluation/registration.json').exists())
sys.path.insert(0, str(ROOT))
from scripts.recover_evidence import validate_public
evidence = json.loads((ROOT/"public_results/independent-receipts.json").read_text())
smoke = json.loads((ROOT/"public_results/live-smoke.json").read_text())
assert smoke["status"] == "passed"
expected = {smoke["checks"][key]["body"]["receipt_id"] for key in ("ordinary", "safety")}
observed = {row["receipt_id"] for row in evidence["rows"]}
assert expected == observed
for row in evidence["rows"]:
    assert row["action"] == "send_to_analyst"
validate_public(evidence)
print(json.dumps({"receipts_verified": len(observed), "evidence_mode": evidence["mode"]}, indent=2))


Independent database reads verify stored receipts for authored requests. This does not establish fitted-model behavior, canary performance or a live kill-switch drill.
